Stencil system

Install the stencil_lib wheel using pip 

In [33]:
import subprocess, sys, glob, pathlib

# location of .whl file
_search_paths = [
    pathlib.Path(__file__).parent if "__file__" in dir() else pathlib.Path("."),
    pathlib.Path("../../build/dist"),
]
_wheel = next(
    (str(w) for p in _search_paths for w in p.glob("stencil_lib-*.whl")),
    None
)
if _wheel is None:
    raise FileNotFoundError("stencil_lib wheel not found. Run 'inv build' or place the wheel alongside the notebook.")

# pip install the wheel
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       _wheel, "--force-reinstall"])


print(f"stencil_lib installed from: {_wheel}")


stencil_lib installed from: ..\..\build\dist\stencil_lib-0.1.0-py3-none-any.whl


Start using the stencil_lib library

In [34]:
# Start using the library
from stencil_lib import CipherConfig

In [35]:
class AESConfig(CipherConfig):
    algo = "aes"
    def __init__(self, key: bytes, mode: int, **mode_params):
        self.parameters = {
            "key": key,
            "mode": mode,
            "mode_params": mode_params
        }
    def encrypt(self, plaintext: bytes, *args, **kwargs) -> bytes:
        from Crypto.Cipher import AES
        cipher = AES.new(self.parameters["key"], self.parameters["mode"], **self.parameters["mode_params"])
        return cipher.encrypt(plaintext)
    def decrypt(self, ciphertext: bytes, *args, **kwargs) -> bytes:
        from Crypto.Cipher import AES
        cipher = AES.new(self.parameters["key"], self.parameters["mode"], **self.parameters["mode_params"])
        return cipher.decrypt(ciphertext)


In [36]:
class CaesarConfig(CipherConfig):
    algo = "caesar"
    def __init__(self, shift: int):
        self.parameters = {
            "shift": shift
        }
    def encrypt(self, plaintext: bytes, *args, **kwargs) -> bytes:
        shift = self.parameters["shift"]
        return bytes((b + shift) % 256 for b in plaintext)
    def decrypt(self, ciphertext: bytes, *args, **kwargs) -> bytes:
        shift = self.parameters["shift"]
        return bytes((b - shift) % 256 for b in ciphertext)


In [37]:
class VigenereConfig(CipherConfig):
    algo = "vigenere"
    def __init__(self, keyword: bytes):
        self.parameters = {
            "keyword": keyword
        }
    def encrypt(self, plaintext: bytes, *args, **kwargs) -> bytes:
        key = self.parameters["keyword"]
        key_len = len(key)
        return bytes((b + key[i % key_len]) % 256 for i, b in enumerate(plaintext))
    def decrypt(self, ciphertext: bytes, *args, **kwargs) -> bytes:
        key = self.parameters["keyword"]
        key_len = len(key)
        return bytes((b - key[i % key_len]) % 256 for i, b in enumerate(ciphertext))


# Stencil_system demo

The stencil library renders 3 APIs namely keygen, encrypt and decrypt

The usage of these APIs are demonstarated below.

In [38]:
import random
import stencil_lib


grid_shape = stencil_lib.GridShape(n=2, shape=(12, 12))
in_byte_len = 16
plaintext = random.randbytes(in_byte_len)
num_partitions = 4
print(f"Plaintext : {' '.join(f'{b:02x}' for b in plaintext)}")

# cipher_cfg = AESConfig(key=random.randbytes(16), mode=1, iv=random.randbytes(16))
# cipher_cfg = CaesarConfig(shift=13)
cipher_cfg = VigenereConfig(keyword=b"KEY")

# --- New: create StencilConfig ---
stencil_cfg = stencil_lib.StencilConfig(
    total_bytes=in_byte_len,
    num_partitions=num_partitions,
    grid_shape=grid_shape,
)

Plaintext : 8b f2 32 4e 06 28 63 04 d6 0c 1c f9 a2 26 54 ab


In [39]:
# Grid generation is optional. If Grid is none, 
# it is randomly generated internally inside encrypt API.
grid = stencil_lib.generate_random_grid(stencil_cfg.grid_shape)

rows, cols = grid_shape.shape
print("Initial Grid:")
for i in range(rows):
    row_bytes = [grid.get_value((i, j)) for j in range(cols)]
    hex_bytes = " ".join(f"{b:02x}" for b in row_bytes)
    print(f"  row {i:2d}: {hex_bytes}")

Initial Grid:
  row  0: 2b ff 46 30 11 17 aa 04 74 7d d1 e1
  row  1: 35 61 7b 28 b9 65 69 8d 30 49 d6 9e
  row  2: 55 de bd 52 82 18 ba 3d f4 02 af f4
  row  3: eb 4c ff 64 21 ff bd 66 0f f1 06 fe
  row  4: 18 80 5f 09 4a 7c c9 e3 02 da 9d 37
  row  5: f5 38 7d 37 94 39 2d a5 97 6a 9a c3
  row  6: 79 3d 3c a1 dc 8b 72 4c b8 f0 47 5f
  row  7: 20 13 36 4b 9d ae 37 86 70 0e 44 8c
  row  8: 1e 19 4e 7b 25 d2 24 0b 44 11 7c 22
  row  9: d3 89 14 e2 f8 fb 91 68 bc c1 fb e3
  row 10: e9 7a 75 56 4c 67 cf 29 0c 3d 48 9b
  row 11: 09 75 1d 44 06 43 73 03 d2 7a 36 d3


Keygen API

returns Secret key := stencil_lib.SecretKey type

In [40]:
secret_key = stencil_lib.keygen(
    stencil_cfg,
    cipher_cfg=cipher_cfg,
)

print("Secret Key")
print(f"  Cipher    : {secret_key.cipher_cfg.algo}")
print(f"  Partitions: {secret_key.partition_list}  ({len(secret_key.partition_list)} total, sum={sum(secret_key.partition_list)} bytes)")
print(f"  Stencils  : {len(secret_key.stencils)}")
for i, s in enumerate(secret_key.stencils):
    coords_str = ", ".join(f"({r},{c})" for r, c in s.coords)
    print(f"    [{i}] shape={s.shape!r}  len={s.len}  coords=[{coords_str}]")


Secret Key
  Cipher    : vigenere
  Partitions: [4, 1, 2, 9]  (4 total, sum=16 bytes)
  Stencils  : 4
    [0] shape='skewconnected'  len=4  coords=[(11,0), (10,1), (9,2), (8,1)]
    [1] shape='skewconnected'  len=1  coords=[(0,8)]
    [2] shape='skewconnected'  len=2  coords=[(0,7), (0,6)]
    [3] shape='skewconnected'  len=9  coords=[(8,8), (7,9), (8,9), (9,8), (10,9), (10,10), (9,9), (9,10), (8,10)]


Encrypt API 

returns obfuscated_grid := stencil_lib.Grid type

In [41]:
obfuscated_grid = stencil_lib.encrypt(plaintext, secret_key, stencil_cfg.grid_shape, grid)
print("\nWith Encryption API call, (Cipher text embedded) Obfuscated Grid is ready")



With Encryption API call, (Cipher text embedded) Obfuscated Grid is ready


Print Cipher text and Obfuscated grid for Demo purpose:

In [42]:
from IPython.display import display, HTML

# Just for Demo purpose
ciphertext = cipher_cfg.encrypt(plaintext)

print(f"Ciphertext: {' '.join(f'{b:02x}' for b in ciphertext)}")

PART_COLORS = ["#f0a500", "#4fc3f7", "#81c784", "#f06292", "#ce93d8", "#80cbc4"]

# --- Partitioned ciphertext ---
parts_lines = []
offset = 0
for i, length in enumerate(secret_key.partition_list):
    chunk = ciphertext[offset: offset + length]
    color = PART_COLORS[i % len(PART_COLORS)]
    hex_part = " ".join(f"{b:02x}" for b in chunk)
    parts_lines.append(
        f'  P{i} ({length:2d}B): <span style="color:{color};font-weight:bold">{hex_part}</span>'
    )
    offset += length

display(HTML(
    "<b>Ciphertext — by partition</b>"
    '<pre style="line-height:1.8">' + "\n".join(parts_lines) + "</pre>"
))

# --- Obfuscated grid: all bytes of a partition share its color; first byte underlined ---
coord_to_part  = {(r, c): i for i, s in enumerate(secret_key.stencils) for r, c in s.coords}
first_coords   = {s.coords[0] for s in secret_key.stencils}

rows_html = []
rows, cols = stencil_cfg.grid_shape.shape
for i in range(rows):
    cells = []
    for j in range(cols):
        b = obfuscated_grid.get_value((i, j))
        token = f"{b:02x}"
        if (i, j) in coord_to_part:
            part_idx = coord_to_part[(i, j)]
            color = PART_COLORS[part_idx % len(PART_COLORS)]
            underline = ";text-decoration:underline" if (i, j) in first_coords else ""
            token = f'<span style="color:{color};font-weight:bold{underline}">{token}</span>'
        cells.append(token)
    rows_html.append(f'  row {i:2d}: {' '.join(cells)}')

legend = "  ".join(
    f'<span style="color:{PART_COLORS[i % len(PART_COLORS)]};font-weight:bold">P{i}</span>'
    for i in range(len(secret_key.stencils))
)
display(HTML(
    f"<b>Obfuscated Grid</b> — {legend} (underlined = partition start)<br>"
    '<pre style="line-height:1.6">' + "\n".join(rows_html) + "</pre>"
))


Ciphertext: d6 37 8b 99 4b 81 ae 49 2f 57 61 52 ed 6b ad f6


Decryprt API 

returns plaintext := bytes type

In [43]:
recovered = stencil_lib.decrypt(obfuscated_grid, secret_key)
assert recovered == plaintext

print(f"Decrypted Plaintext : {' '.join(f'{b:02x}' for b in recovered)}")

Decrypted Plaintext : 8b f2 32 4e 06 28 63 04 d6 0c 1c f9 a2 26 54 ab
